# Kaggle Resume Orchestrator (CPU-only)

**Purpose:** watches `sham_small_training.ipynb`'s real Kaggle kernel and automatically pushes a fresh run whenever it has stopped (finished its ~8.5h budget, errored, or was cancelled) — so training resumes without anyone manually reopening it and clicking "Save & Run All" again.

**Why this is a SEPARATE, CPU-only notebook instead of scheduling the training notebook itself:** Kaggle's own native "Schedule this notebook" feature refuses to schedule any notebook with a GPU accelerator attached. This one has **no accelerator at all** — set it to `None` when you upload/configure it — specifically so Kaggle's own scheduler will accept it. It never trains anything itself; it only checks status and re-triggers the real GPU notebook via the Kaggle API.

**Revision note (2026-09-20):** an earlier attempt drove this same check-and-resume logic from a GitHub Actions workflow instead. That failed in practice (the push did not reliably take effect and training then failed at startup) and has been removed. This notebook reuses the exact same, already-tested `resume_kernel()` decision logic — see `ai-system/scripts/kaggle_auto_resume.py`'s own module docstring and self-test — just called from inside Kaggle's own infrastructure instead of from GitHub.

**One-time setup, before scheduling this:**
1. Set the accelerator for THIS notebook to **None** (not the training notebook — leave that one's GPU as-is).
2. Turn **Internet On** for this notebook.
3. Attach the same 3 secrets already used by `sham_small_training.ipynb` (Add-ons → Secrets): `GITHUB_TOKEN`, `KAGGLE_USERNAME`, `KAGGLE_KEY`.
4. Edit `TARGET_KERNEL_SLUG` in the cell below to your real training kernel's slug (shown in its own Kaggle URL, e.g. `jonsnowjonsnow/notebook2d0e40c1f1`).
5. Save this notebook, then use Kaggle's own **"Schedule this notebook"** button (Settings) — e.g. every 9 hours, matching the training notebook's own `MAX_TRAINING_HOURS`.

**Known platform caveat** (unchanged from before, not something this notebook can fix): a `kaggle kernels push` can disconnect the pushed kernel's own attached Secrets even when it reports success. Check `sham_small_training.ipynb`'s own Secrets tab after the first few automated resumes to confirm they held.

In [ ]:
# EDIT THIS before scheduling -- never left as a guess, since a wrong
# slug would resume/duplicate the wrong kernel entirely.
TARGET_KERNEL_SLUG = "REPLACE_ME/your-training-kernel-slug"

assert TARGET_KERNEL_SLUG != "REPLACE_ME/your-training-kernel-slug", (
    "Set TARGET_KERNEL_SLUG above to your real training kernel's slug "
    "(visible in its own Kaggle URL) before scheduling this notebook."
)

In [ ]:
import os
import subprocess
from kaggle_secrets import UserSecretsClient

# Same clone/pull pattern as sham_small_training.ipynb's own first cell
# -- this notebook always runs against the LATEST kaggle_auto_resume.py
# in the repo, never a stale copy pasted in here by hand.
GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)

subprocess.run(["pip", "install", "-q", "-U", "kaggle"], check=True)
print("repo ready, kaggle package installed.")

In [ ]:
import os
import sys
from kaggle_secrets import UserSecretsClient

# Same Kaggle-Secrets pattern already used by sham_small_training.ipynb's
# own checkpoint-publish cell -- no new secret needed, this reuses the
# exact two that already exist for that purpose.
KAGGLE_USERNAME = UserSecretsClient().get_secret("KAGGLE_USERNAME")
KAGGLE_KEY = UserSecretsClient().get_secret("KAGGLE_KEY")
os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = KAGGLE_KEY

sys.path.insert(0, "/kaggle/working/Ttbik/ai-system/scripts")
from kaggle_auto_resume import resume_kernel

result = resume_kernel(
    kernel_slug=TARGET_KERNEL_SLUG,
    notebook_repo_path="/kaggle/working/Ttbik/ai-system/colab/sham_small/kaggle_notebooks/sham_small_training.ipynb",
    work_dir="/kaggle/working/kaggle-kernel",
    sync_from_repo=False,
)
print(f"\nresult: {result}")